# 02. Baselines, validation and exploratory analysis

UTS 43008 · Assignment 3 · Project 12 · Dihedrals

Before asking whether PPO learns, check that the game is reproducible and that the comparison
policies behave as intended. This notebook uses small development samples. Its plots are
diagnostics, not final assignment results.

How strong is the scripted commander? How variable is valid-random play? Which layouts survive
the planned difficulty filter?

In [ ]:
from pathlib import Path
import os
import sys
import json
import uuid
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

candidate = Path(os.environ.get("BASE_SIEGE_ROOT", Path.cwd())).expanduser()
ROOT = next((p for p in [candidate, *candidate.parents] if (p / "env/siege_env.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the complete project, or set BASE_SIEGE_ROOT to its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from env import config as C
from env.siege_env import SiegeEnv, describe_action, deployment_action
from env.render import draw_snapshot
from env.generator import generate_base
from experiments.provenance import runtime_manifest, sha256
RUNS = Path(os.environ.get("BASE_SIEGE_RUNS_DIR", ROOT / "runs")).expanduser().resolve()
OUTPUT = ROOT / "artifacts" / "notebook-sessions" / uuid.uuid4().hex[:12]
OUTPUT.mkdir(parents=True, exist_ok=False)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 12, "axes.labelsize": 10, "font.size": 10})
def save_figure(name):
    plt.savefig(OUTPUT / f"{name}.png", dpi=180, bbox_inches="tight")
    plt.show()
print(f"Project: {ROOT.name} | Python: {sys.version.split()[0]}")
print(f"This notebook's outputs: {OUTPUT}")

## 1. Two policies with different purposes

Valid-random samples legal actions only. It does not sample illegal actions and receive an
invented penalty. The heuristic buys a fixed 199-resource army, chooses a side using defence
coverage and follows a scripted deployment order.

Neither policy learns. PPO must be compared with both on identical layouts and views.

In [ ]:
from agents.heuristic import HeuristicCommander
from agents.random_agent import RandomAgent
from experiments.rollout import play_episode
from time import perf_counter

army = pd.Series(C.HEURISTIC_ARMY, name="count")
display(pd.DataFrame({"count": army, "unit_cost": pd.Series(C.TROOP_COSTS)}))
print("Army cost:", sum(C.HEURISTIC_ARMY[t] * C.TROOP_COSTS[t] for t in C.TROOP_TYPES))

## 2. Matched development episodes

Use six seed/structure pairs and three random action seeds. Repeated random seeds describe
action variability; they are not independent trained learners. Save raw rows before averaging.

In [ ]:
rows = []
for base_seed in range(10200, 10206):
    for label, action_seed in [("heuristic", -1)] + [("random", s) for s in range(3)]:
        policy = HeuristicCommander() if label == "heuristic" else RandomAgent(action_seed)
        board = SiegeEnv(base_seed=base_seed, motifs=("closed_ring",))
        started = perf_counter()
        result = play_episode(board, policy)["summary"]
        rows.append({"agent": label, "base_seed": base_seed, "action_seed": action_seed,
            "destruction": result["destruction"], "return": result["episode_return"],
            "spent": result["spent"], "ticks": result["tick"], "finish_reason": result["finish_reason"],
            "seconds": perf_counter() - started})
        board.close()
results = pd.DataFrame(rows)
results.to_csv(OUTPUT / "baseline-development-episodes.csv", index=False)
display(results.head(8))

In [ ]:
display(results.groupby("agent").agg(episodes=("base_seed", "size"),
    mean_destruction=("destruction", "mean"), std_destruction=("destruction", "std"),
    mean_return=("return", "mean"), mean_spent=("spent", "mean"), seconds=("seconds", "sum")))
display(pd.crosstab(results.agent, results.finish_reason))

## 3. Look beyond a single mean

The left plot compares policies on matching layouts. The right shows the spread of episode
outcomes. Six layouts do not establish general performance. These unfiltered demonstration
bases are a different population from the filtered study pool.

In [ ]:
per_base = results.groupby(["base_seed", "agent"]).destruction.mean().unstack()
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
per_base.plot.bar(ax=axes[0], rot=0)
axes[0].set(xlabel="Development layout seed", ylabel="Mean destruction", ylim=(0, 1))
for label, group in results.groupby("agent"):
    values = np.sort(group.destruction.to_numpy())
    axes[1].step(values, np.arange(1, len(values)+1)/len(values), where="post", label=label)
axes[1].set(xlabel="Episode destruction", ylabel="Fraction of episodes at or below value", xlim=(0, 1))
axes[1].legend()
plt.tight_layout()
save_figure("baseline-diagnostics")

## 4. Test determinism

An identical layout, policy and action seed should reproduce the actions and outcome. Runtime
measurements are not deterministic and are deliberately excluded from the equality check.

In [ ]:
def diagnostic_episode(action_seed):
    board = SiegeEnv(base_seed=10200, motifs=("closed_ring",))
    try:
        return play_episode(board, RandomAgent(action_seed))
    finally:
        board.close()
first, repeat = diagnostic_episode(17), diagnostic_episode(17)
assert first == repeat
display(pd.Series({"actions": len(first["actions"]), "same_seed_identical": first == repeat,
    "different_seed_actions_differ": first["actions"] != diagnostic_episode(18)["actions"]}))

## 5. Structural validity is not difficulty

Structural checks reject overlaps, buildings on walls, incorrect building counts, an unprotected
Town Hall or empty required compartments.

The difficulty test attacks each valid base from four sides with a heuristic and takes the best
destruction fraction. The study keeps values from 0.25 to 0.85 inclusive. That four-side maximum
is a layout-filter rule, not the heuristic's reported baseline score.

In [ ]:
from experiments.pools import validate_structure, validate_base
checks = []
for base_seed in range(12):
    spec = generate_base(base_seed, "closed_ring")
    structural, _ = validate_structure(spec)
    accepted, score, reason = validate_base(spec)
    checks.append({"base_seed": base_seed, "structural": structural, "accepted": accepted,
                   "best_side_destruction": score, "reason": reason})
filter_results = pd.DataFrame(checks)
display(filter_results)
display(filter_results.groupby("accepted").size().rename("candidate_count"))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.axhspan(C.DIFFICULTY_MIN, C.DIFFICULTY_MAX, alpha=.12, color="green", label="Accepted interval")
ax.scatter(filter_results.base_seed, filter_results.best_side_destruction,
           c=np.where(filter_results.accepted, "#2a8065", "#b14c45"))
ax.set(xlabel="Candidate layout seed", ylabel="Best four-side heuristic destruction", ylim=(0, 1))
ax.legend()
save_figure("difficulty-filter")

## 6. The filter changes the population

A result on accepted bases is not evidence that the policy handles every generated base.
Report candidate counts, acceptance rates and rejection reasons by structure. Do not keep
sampling until a preferred policy gets an attractive score.

Small diagnostic pools are for software checks. Full pools must be built and frozen separately.

In [ ]:
display(pd.DataFrame([
    {"partition": "training", "first": C.TRAIN_SEED_RANGE[0], "last": C.TRAIN_SEED_RANGE[1], "purpose": "learn parameters"},
    {"partition": "development", "first": C.DEV_SEED_RANGE[0], "last": C.DEV_SEED_RANGE[1], "purpose": "choose the setup"},
    {"partition": "final", "first": C.FINAL_STUDY_SEED_RANGE[0], "last": C.FINAL_STUDY_SEED_RANGE[1], "purpose": "one frozen evaluation"}
]))
print("Ranges only. No final layouts were generated.")

## 7. Regression checks and viva questions

Run the full suite with the command below. It checks mechanics, masks, reward charging, D4
mappings and protected seed ranges. Passing tests is not evidence of a good learned policy.

~~~bash
uv run pytest -q
~~~

Explain the difference between a layout seed, action seed and learner seed. Why can ten random
action seeds not replace five independent PPO training runs?